In [2]:
"""
Titanic Data Analysis and JSON Export
Author: Francesco
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


In [5]:
df = pd.read_csv(CSV_FILE)
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [6]:
# Select numeric columns only
numeric_columns = df.select_dtypes(include="number")

# Show which columns were selected
print(f"Numeric columns: {list(numeric_columns.columns)}")

# Calculate the statistics
statistics = pd.DataFrame({
    "Mean": numeric_columns.mean(),
    "Median": numeric_columns.median(),
    "Standard deviation": numeric_columns.std()
})

# Display the results, rounded to 2 decimal places
print(statistics.round(2))

Numeric columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
               Mean  Median  Standard deviation
PassengerId  446.00  446.00              257.35
Survived       0.38    0.00                0.49
Pclass         2.31    3.00                0.84
Age           29.70   28.00               14.53
SibSp          0.52    0.00                1.10
Parch          0.38    0.00                0.81
Fare          32.20   14.45               49.69


In [9]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
 
missing_data = {}
 
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    if missing_count > 0:
        missing_data[col] = {
            "Missing count": missing_count,
            "Missing percent": missing_percent
        }

if missing_data:
        missing_table = pd.DataFrame.from_dict(missing_data, orient="index")
        print(missing_table.round(2))
else:
        print("No missing values found.")


MISSING VALUES ANALYSIS
          Missing count  Missing percent
Age                 177            19.87
Cabin               687            77.10
Embarked              2             0.22


In [12]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()
 
# Feature 1: Family Size
df_features["FamilySize"] = df_features["SibSp"] + df_features["Parch"] + 1
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))
 
# Feature 2: Is Alone
df_features["IsAlone"] = (df_features["FamilySize"] == 1).astype(int)
print(df_features[['FamilySize', 'IsAlone']].head(10))
 
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Minor'
    elif age < 30:
        return 'Under 30'
    elif age < 50:
        return 'Middle Aged'
    else:
        return 'Elderly'
 
df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))
 
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Here is an example to get you started:
print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)
 
# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0
    Age     AgeGroup
0  22.0     Under 30
1  38.0  Middle Aged
2  26.0     Under 30
3  35.0  Middle Aged
4  35.0  Middle Aged
5   NaN      Unknown
6  54.0      Elderly
7   2.0        Minor
8  27.0     Under 30
9  14.0        Minor

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0 

In [13]:
print("\nSurvival by Passenger Class:")

class_survival = (
    df_features.groupby("Pclass")["Survived"]
    .agg(["mean", "count"])
)

class_survival["Survival rate (%)"] = class_survival["mean"] * 100

print(class_survival[["count", "Survival rate (%)"]].round(1))


Survival by Passenger Class:
        count  Survival rate (%)
Pclass                          
1         216               63.0
2         184               47.3
3         491               24.2


In [14]:
from datetime import datetime

class Passenger:
    """Represents one passenger with their information."""

    def __init__(
        self, passenger_id, name, age, sex, survived, pclass,
        fare, embarked=None, family_size=None, is_alone=None, title=None
    ):
        # Convert present values to simple Python types; use None for missing values
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = bool(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None

    def to_dict(self):
        """Convert this passenger to a dictionary."""
        return {
            "passenger_id": self.passenger_id,
            "name": self.name,
            "age": self.age,
            "sex": self.sex,
            "survived": self.survived,
            "pclass": self.pclass,
            "fare": self.fare,
            "embarked": self.embarked,
            "family_size": self.family_size,
            "is_alone": self.is_alone,
            "title": self.title
        }


class TitanicDataset:
    """Represents the full dataset and provides JSON export."""

    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []
        self._create_passengers()

    def _create_passengers(self):
        """Create one Passenger object for each dataframe row."""
        for idx, row in self.dataframe.iterrows():
            name = row.get("Name")
            title = row.get("Title")

            # The CSV has no Title column, so get the title from Name
            if pd.isna(title) and pd.notna(name):
                name_parts = str(name).split(",", 1)
                if len(name_parts) > 1 and "." in name_parts[1]:
                    title = name_parts[1].split(".", 1)[0].strip()

            passenger = Passenger(
                passenger_id=row.get("PassengerId", idx),
                name=name,
                age=row.get("Age"),
                sex=row.get("Sex"),
                survived=row.get("Survived"),
                pclass=row.get("Pclass"),
                fare=row.get("Fare"),
                embarked=row.get("Embarked"),
                family_size=row.get("FamilySize"),
                is_alone=row.get("IsAlone"),
                title=title
            )

            self.passengers.append(passenger)

    def get_summary_stats(self):
        """Calculate summary statistics for the passengers."""
        total_passengers = len(self.passengers)
        survived_count = sum(
            1 for passenger in self.passengers if passenger.survived == 1
        )
        did_not_survive_count = sum(
            1 for passenger in self.passengers if passenger.survived == 0
        )

        ages = [
            passenger.age for passenger in self.passengers
            if passenger.age is not None
        ]
        fares = [
            passenger.fare for passenger in self.passengers
            if passenger.fare is not None
        ]

        survival_rate = 0
        if total_passengers > 0:
            survival_rate = survived_count / total_passengers

        average_age = None
        if ages:
            average_age = sum(ages) / len(ages)

        average_fare = None
        if fares:
            average_fare = sum(fares) / len(fares)

        return {
            "total_passengers": total_passengers,
            "survived": survived_count,
            "did_not_survive": did_not_survive_count,
            "survival_rate": round(survival_rate, 4),
            "average_age": round(average_age, 2) if average_age is not None else None,
            "average_fare": round(average_fare, 2) if average_fare is not None else None
        }

    def to_json(self, filename="titanic_data.json"):
        """Export the dataset to a JSON file."""
        summary = self.get_summary_stats()

        data = {
            "metadata": {
                "dataset_name": "Titanic Passenger Dataset",
                "export_date": datetime.now().isoformat(),
                "total_passengers": summary["total_passengers"],
                "survived": summary["survived"],
                "did_not_survive": summary["did_not_survive"],
                "survival_rate": summary["survival_rate"],
                "average_age": summary["average_age"],
                "average_fare": summary["average_fare"]
            },
            "passengers": [
                passenger.to_dict() for passenger in self.passengers
            ]
        }

        with open(filename, "w", encoding="utf-8") as json_file:
            json.dump(data, json_file, indent=2, ensure_ascii=False)

        print(f"Data exported to {filename}")
        return data


# Use the dataframe copy made in the feature-engineering step
if "df_features" in globals() and not df_features.empty:
    dataset = TitanicDataset(df_features)

    print(f"Created dataset with {len(dataset.passengers)} passengers.")
    print("\nSummary statistics:")

    summary_stats = dataset.get_summary_stats()
    for name, value in summary_stats.items():
        print(f"{name}: {value}")

    dataset.to_json(JSON_FILE)
else:
    print("df_features is missing or empty. Run the feature-engineering cell first.")

Created dataset with 891 passengers.

Summary statistics:
total_passengers: 891
survived: 342
did_not_survive: 549
survival_rate: 0.3838
average_age: 29.7
average_fare: 32.2
Data exported to data/titanic_data.json


In [15]:
# Load the exported JSON file
with open(JSON_FILE, "r", encoding="utf-8") as f:
    json_data = json.load(f)

# Show the top-level structure and metadata
print("Top-level keys:", list(json_data.keys()))
print("Metadata:", json_data["metadata"])

# Check how many passenger records were saved
passenger_records = json_data["passengers"]
print("Passenger records:", len(passenger_records))
print("Dataframe rows:", len(df_features))

# Display one passenger record in readable form
print("\nFirst passenger record:")
print(json.dumps(passenger_records[0], indent=2, ensure_ascii=False))

# Check that the exported record count matches the dataframe
if len(passenger_records) == len(df_features):
    print("\nValidation passed: passenger counts match.")
else:
    print("\nValidation failed: passenger counts do not match.")

Top-level keys: ['metadata', 'passengers']
Metadata: {'dataset_name': 'Titanic Passenger Dataset', 'export_date': '2026-11-05T13:22:25.915780', 'total_passengers': 891, 'survived': 342, 'did_not_survive': 549, 'survival_rate': 0.3838, 'average_age': 29.7, 'average_fare': 32.2}
Passenger records: 891
Dataframe rows: 891

First passenger record:
{
  "passenger_id": 1,
  "name": "Braund, Mr. Owen Harris",
  "age": 22.0,
  "sex": "male",
  "survived": 0,
  "pclass": 3,
  "fare": 7.25,
  "embarked": "S",
  "family_size": 2,
  "is_alone": false,
  "title": "Mr"
}

Validation passed: passenger counts match.
